# pyspi

pyspi computes **statistics of pairwise interactions** (SPIs) between the processes of a multivariate time series. Each SPI is one answer to "how are these two processes related?" — a correlation, a spectral coherence, a transfer entropy, a causal score. For `M` processes each returns an `M x M` matrix, and pyspi stacks them so hundreds of notions of "related" can be compared side by side.

## 1. Data

In [ ]:
import numpy as np
from pyspi.calculator import Calculator, bundled_configs
from pyspi.data import load_dataset, available_datasets

available_datasets()

`Data` holds an `M x T` array, z-scored per process by default. Your own data goes in the same way — `Calculator(dataset=my_array)` accepts a NumPy array, and `dim_order` controls whether rows are processes or observations.

In [ ]:
data = load_dataset("cml")   # coupled map lattice: 10 processes, 500 observations
print(data.n_processes, "processes x", data.n_observations, "observations")

## 2. Compute

`config` selects the SPI set. `fabfour` is the smallest (4 SPIs, one per family); `full` is all 325. See `bundled_configs()`.

In [ ]:
calc = Calculator(dataset=data, config="fabfour", verbose=False)
calc.compute(progress=False)

calc.summary()

## 3. Results

`calc.table` is wide: rows are processes, columns a `(spi, process)` MultiIndex. Entry `[i, j]` is computed with `i` as **source** and `j` as **target** — irrelevant for a symmetric SPI, the whole point for a directed one.

In [ ]:
calc.table["cov_EmpiricalCovariance"].round(3)

`to_frame()` gives the same data in long form, one row per `(spi, source, target)` — easier to plot, group, or join against SPI labels.

In [ ]:
long = calc.to_frame()
long.groupby("spi")["value"].agg(["mean", "std"]).round(3)

### Direction

The data is z-scored, so covariance *is* Pearson correlation and its matrix is symmetric. Directed information is not.

In [ ]:
di = calc.table["di_gaussian_n-5"]
print("0 -> 1:", round(di.iloc[0, 1], 3), " 1 -> 0:", round(di.iloc[1, 0], 3))
print("symmetric?", np.allclose(di.values, di.values.T, equal_nan=True))

## 4. Cost

Cost is heavily skewed: a few SPIs dominate, most are effectively free. `summary()` reports the total and the slowest five, which is how to budget a run.

The `benchmarked_p80/p90/p95/p99` configs exploit that skew — they keep the fastest N% by measured cost. To build your own subset by keyword, `filter_spis(["directed", "nonlinear"], output_name="mine")` writes a config you pass as `Calculator(config="mine.yaml")`.

In [ ]:
import time

t0 = time.perf_counter()
sonnet = Calculator(dataset=data, config="sonnet", verbose=False)   # 14 SPIs, one per family
sonnet.compute(progress=False)
print(f"{sonnet.n_spis} SPIs in {time.perf_counter() - t0:.1f}s")

sonnet.summary()["slowest"]

## 5. Failures

A failed SPI still occupies its column, filled with NaN. Check `calc.errors` before reading a table: a NaN column is otherwise indistinguishable from a legitimately undefined statistic.

In [ ]:
sonnet.errors or "no failures"

## 6. Saving, scale, CLI

`.npz` round-trips exactly and needs nothing but numpy; `.csv` is a one-way export.

```python
calc.save("results.npz")
from pyspi.calculator import load_table
table = load_table("results.npz")
```

**Many datasets: run one per process** (a cluster array job, or GNU parallel) rather than raising `n_jobs`. SPIs sharing a cache are grouped into one sequential task, so within-dataset speedup is floored by the longest group — measured at 2.3–4.5x whatever `n_jobs` you pass. One dataset per process scales close to linearly and confines a failure to one dataset.

Everything above is available without writing Python:

```bash
python -m pyspi compute --data ts.npy --config fabfour --checkpoint-dir results/
```

`--checkpoint-dir` writes each SPI as it finishes, so an interrupted run resumes instead of restarting.